In [2]:
# !/usr/bin/env python3
import os
import torch
from equiv_dens.training.parse_command_line_arguments import parse_command_line_arguments
from equiv_dens.data.density_dataset import AtomsDensityData
from equiv_dens.utils.grids import cubical_grid, cubical_sampling,\
    spherical_grid, spherical_radial_sampling
import equiv_dens.utils.base as utils
from equiv_dens.training.model_loader import load_model

import numpy as np
from functools import partial
import argparse
%load_ext autoreload
%autoreload 2

Use "numpy" for Fourier Transform


/home/mihail/anaconda3/envs/equiv_dens/lib/python3.7/site-packages/pyscf/lib/misc.py:47: H5pyDeprecationWarning: Using default_file_mode other than 'r' is deprecated. Pass the mode to h5py.File() instead.
  h5py.get_config().default_file_mode = 'a'


In [3]:
class LoadFromFile (argparse.Action):
    def __call__ (self, parser, namespace, values, option_string = None):
        with values as f:
            # parse arguments in the file and store them in the target namespace
            parser.parse_args(f.read().split(), namespace)

In [39]:
args, hyperparam_args = parse_command_line_arguments(arg_file='ethanol_dens_010_mae_en_only_scaling_001_test.txt')
print('type dtype', type(args.dtype))
args.fix_parameters = True

print('args np dir', args.np_dataset)
# no restart directory specified
directory = args.restart  # load directory name
# load latest checkpoint
checkpoint_path = os.path.join(directory, 'checkpoints')  # checkpoint directory
checkpoint = torch.load(os.path.join(
    checkpoint_path, 'latest_checkpoint.pth'), map_location='cpu')
latest_checkpoint = checkpoint['step']
model_code = checkpoint['ID']  # load ID
step = checkpoint['step']
for arg in vars(checkpoint['args']):
    if args.fix_arguments:
        if arg in hyperparam_args:
            print('loading hyperparam arg', arg)
            setattr(args, arg, getattr(checkpoint['args'], arg))
    else:
        print('loading all arg', arg)
        setattr(args, arg, getattr(checkpoint['args'], arg))
restore = True

args.best_model_path = 'best_' + model_code + '.pth'
print('best_model_path', args.best_model_path)

print('model code:', model_code)
# determine whether GPU is used for training
print('args use gpu', args.use_gpu)
args.use_gpu = args.use_gpu and torch.cuda.is_available()

# load dataset(s)
print("loading density from" + str(args.dens_dataset) + "...")
print("loading atoms from" + args.np_dataset + "...")
args.use_gpu = False
args.spherical_grid_level = 4
if args.cube_grid:
    grid_origin = args.cube_origin
    grid_extent = np.array([args.cube_extent] * 3)
    grid_fn = partial(cubical_grid, nx=args.cube_size, ny=args.cube_size, nz=args.cube_size,
                      extent=grid_extent,
                      origin=np.array([grid_origin] * 3))
    sampling_fn = cubical_sampling
else:
    grid_fn = partial(spherical_grid, level=args.spherical_grid_level)
    sampling_fn = partial(spherical_radial_sampling, rotate=False)
    grid_origin = 0
    grid_extent = None

dataset = AtomsDensityData(np_path=args.np_dataset, density_path=args.dens_dataset,
                           orbitals_path=args.orbitals_file,
                           density_n_samp=10000000000,
                           required_properties=['density'],
                           center_positions=False,
                           radial_coeffs_file=args.radial_coeffs_file,
                           dtype=args.dtype,
                           grid_fn=grid_fn,
                           sampling_fn=sampling_fn,
                           grid_extent=grid_extent,
                           grid_origin=grid_origin,
                           verbose=args.verbose)

model = load_model(args, dataset)

type dtype <class 'torch.dtype'>
args np dir datasets/ethanol_dft_train.npy
loading hyperparam arg activation
loading hyperparam arg order
loading hyperparam arg mixing_order
loading hyperparam arg order_en
loading hyperparam arg mixing_order_en
loading hyperparam arg num_features
loading hyperparam arg num_basis_functions
loading hyperparam arg num_radial_components
loading hyperparam arg num_energy_features
loading hyperparam arg num_modules
loading hyperparam arg num_residual_pre_x
loading hyperparam arg num_residual_post_x
loading hyperparam arg num_residual_pre_vi
loading hyperparam arg num_residual_pre_vj
loading hyperparam arg num_residual_post_v
loading hyperparam arg num_residual_output
loading hyperparam arg num_energy_output
loading hyperparam arg basis_functions
loading hyperparam arg cutoff
loading hyperparam arg orthonormal_basis
loading hyperparam arg expansion_constraint
loading hyperparam arg integral_constraint
loading hyperparam arg integral_scale
loading hyperparam 

In [40]:
from pyscf.data import nist

np.random.seed(args.split_seed)
# start_ind = 10
print('batch size', args.test_batch_size)
print('density grid level', args.spherical_grid_level)
start_idx = np.random.randint(len(dataset), size=(10,))
print('start idx', start_idx)
r_density_ints = []
d_density_ints = []
r_dpm = []
d_dpm = []
dens_diff = []
model.eval()
for idx in start_idx:
    atoms_data = dataset.get_properties([idx])
    print('positions', atoms_data['positions'])
    total_charge = torch.sum(atoms_data['atom_numbers'][0])
    results = model(atoms_data)
    results_density_integral = torch.sum(results['density'] * results['coord_weights'], dim=1).detach()
    r_density_ints.append(results_density_integral)
    r_dpm.append(results['dipole_moment'].detach())

    dpm_module = model.property_models['dipole_moment']
    print('density shape', results['density'].shape)
    dpm = dpm_module(atoms_data)
    d_dpm.append(dpm['dipole_moment'].detach())
    data_density_integral = torch.sum(atoms_data['density'] * atoms_data['coord_weights'], dim=1).detach()
    d_density_ints.append(data_density_integral)
    
    density_diff = torch.sum(torch.abs(atoms_data['density'] - results['density']) * atoms_data['coord_weights'], dim=1).detach()
    density_diff /= data_density_integral
    dens_diff.append(density_diff)
r_density_ints = torch.cat(r_density_ints)
d_density_ints = torch.cat(d_density_ints)

r_dpm = torch.cat(r_dpm, dim=0) * utils.to_bohr * nist.AU2DEBYE
d_dpm = torch.cat(d_dpm, dim=0) * utils.to_bohr * nist.AU2DEBYE
dens_diff = torch.cat(dens_diff)
print('dipole moment results', r_dpm)
print('dipole moment data', d_dpm)
print('dipole moment magnitude results', torch.sqrt(torch.sum(r_dpm**2, -1)))
print('dipole moment magnitude data', torch.sqrt(torch.sum(d_dpm**2, -1)))
print('results density integrals', r_density_ints)
print('results density integral error', torch.mean(torch.abs(r_density_ints - total_charge)))
print('data density integral error', torch.mean(torch.abs(d_density_ints - total_charge)))
print('data density integrals', d_density_ints)
print('dipole errors', torch.sqrt(torch.sum((r_dpm - d_dpm)**2, dim=1)))
print('mean square dipole error', torch.mean(torch.sqrt(torch.sum((r_dpm - d_dpm)**2, dim=1))))
print('mean absolute dipole error', torch.mean(torch.abs(r_dpm - d_dpm)))
print('mean density error', torch.mean(dens_diff))

batch size 1
density grid level 4
start idx [102 435 860 270 106  71 700  20 614 121]
positions tensor([[[-0.0065,  0.2638,  0.4427],
         [-0.2148,  0.9986, -0.7896],
         [ 0.2604, -1.1347,  0.3659],
         [ 0.7558,  0.6564,  1.1090],
         [-0.9952,  0.3704,  1.1336],
         [ 0.5350,  0.7662, -1.4727],
         [-1.0073,  0.5283, -1.4629],
         [-0.3130,  2.0852, -0.7120],
         [-0.4737, -1.4402, -0.2690]]])
density shape torch.Size([1, 181716])
positions tensor([[[-0.0138, -0.5501,  0.3398],
         [-1.1309,  0.5372,  0.2187],
         [ 1.1106, -0.0315, -0.4359],
         [ 0.2029, -0.7832,  1.4998],
         [-0.2468, -1.5044, -0.2350],
         [-2.1079,  0.3195,  0.5695],
         [-0.8911,  1.5663,  0.5703],
         [-1.5050,  0.7921, -0.8924],
         [ 0.5571,  0.2628, -1.2471]]])
density shape torch.Size([1, 181716])
positions tensor([[[-0.0933, -0.4783,  0.2163],
         [-1.0269,  0.7138,  0.5418],
         [ 1.0580, -0.2548, -0.6430],
      

In [41]:
from pyscf import gto, dft
from pyscf.scf import hf
atom_data = dataset.get_properties(start_idx[:1]) 
atom_types = atom_data['atom_numbers'][0].numpy()
pos = atom_data['positions'][0].numpy()
atom = []
for j in range(len(atom_types)):
    atom.append((atom_types[j], pos[j, :])) 
mol = gto.M(atom=atom, basis='def2svp')
#print(mol.pack())
mf = dft.RKS(mol)
mf.chkfile=False
mf.xc = 'pbe'
mf.max_cycles = -1
mf.kernel()
g = mf.nuc_grad_method()
forces = g.grad()
dpm = mf.dip_moment()
print('dpm', dpm)
print('magnitude', np.sqrt(np.sum(dpm**2)))
print('r_dpm', r_dpm)
print('d_dpm', d_dpm)
print('grid dipole error', np.mean(np.abs(dpm - d_dpm.numpy()[0]))) 
print('grid pred dipole error', np.mean(np.abs(dpm - r_dpm.numpy()[0]))) 

converged SCF energy = -154.699838199348
--------------- RKS gradients ---------------
         x                y                z
0 C     0.0512306165     0.0247414948    -0.0747998507
1 C     0.0346486908     0.0062961053     0.0343863221
2 O     0.0245331945     0.0006468772     0.0381391234
3 H    -0.0160067324    -0.0103202372    -0.0028349051
4 H    -0.0278932491    -0.0028418984     0.0205302542
5 H    -0.0544556189     0.0010946451     0.0281750368
6 H     0.0080261023    -0.0129034109    -0.0199955248
7 H     0.0052805730    -0.0058473502    -0.0025230294
8 H    -0.0253728051    -0.0008617141    -0.0210875436
----------------------------------------------
Dipole moment(X, Y, Z, Debye): -0.97806,  0.79667, -1.06601
dpm [-0.9780557   0.79666726 -1.06600536]
magnitude 1.6515565671517711
r_dpm tensor([[-0.9926,  0.8298, -1.0953],
        [-1.8593,  0.0576, -0.3647],
        [-1.6260,  0.7975, -0.3242],
        [-1.1416,  1.1149, -0.7339],
        [ 1.0168,  0.4816, -1.0530],
    

diff 1 4 tensor(1.9898)
diff 1 4 tensor(1.9899)
diff 1 8 tensor(1.9745)
diff 2 4 tensor(1.5856)
diff 2 8 tensor(1.3412)
diff 4 8 tensor(1.0398)
